# ClinHallu: GPU evaluation of Whisper large-v3 / turbo (Kaggle or Colab, T4)
**Kaggle:** Settings → Accelerator **GPU T4 x1**, Internet **On**. For long runs use **Save Version → Save & Run All (Commit)**, which runs in the background (12 h limit), and download `results_<split>_part<N>.zip` from the version's Output tab.

Set `SPLIT` and `PART` in the next cell. Every step checks for errors and stops with a message if something fails.

In [ ]:
# ---- configuration ----
SPLIT = 'v1'   # 'pilot' (104 sessions) or 'v1' (750 sessions)
PART  = 1      # v1 is split into 2 parts to stay under Kaggle's 12 h session limit
SPLITS = {
  'pilot': '--split pilot --n 4 --gaps 1,12',
  'v1':    '--split v1 --n 30 --seed0 2000 --gaps 1,3,6,12,20 --kinds silence,room,esc:coughing,esc:breathing,esc:keyboard_typing',
}
RUNS = {
  1: [('large-v3','default'), ('large-v3','no_prev'), ('large-v3','vad')],              # ~5 h on T4
  2: [('large-v3-turbo','default'), ('large-v3-turbo','no_prev'), ('large-v3-turbo','vad'), ('large-v3','vad_no_prev'), ('large-v3','vad_pad')],  # ~3.5 h
}[PART] if SPLIT == 'v1' else [('large-v3','default'), ('large-v3','no_prev'), ('large-v3-turbo','default'), ('large-v3-turbo','no_prev')]

In [ ]:
import os, subprocess, sys
def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:], file=sys.stderr)
    if r.returncode != 0: raise RuntimeError(f'FAILED ({r.returncode}): {cmd}')
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
if not os.path.exists('whisper-clinical-hallucination'):
    sh('git clone -q https://github.com/hamzak-27/whisper-clinical-hallucination.git')
os.chdir('whisper-clinical-hallucination'); sh('git log --oneline | head -3')
sh('pip -q install faster-whisper jiwer kokoro-onnx soundfile nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"')

In [ ]:
# Make the pip-installed CUDA libs visible to CTranslate2 (fixes 'libcudnn/libcublas not found')
import glob, site
libdirs = [os.path.dirname(p) for sp in site.getsitepackages() for p in glob.glob(sp + '/nvidia/*/lib/*.so*')]
print('CUDA lib dirs:', sorted(set(libdirs)))
os.environ['LD_LIBRARY_PATH'] = ':'.join(sorted(set(libdirs))) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
# GPU smoke test BEFORE spending time on data
SMOKE = """
import numpy as np
from faster_whisper import WhisperModel
m = WhisperModel("tiny", device="cuda", compute_type="float16")
segs, _ = m.transcribe(np.zeros(16000 * 3, dtype=np.float32))
print(list(segs))
print("GPU OK")
"""
open("/tmp/gpu_smoke.py", "w").write(SMOKE)
sh("python /tmp/gpu_smoke.py")

In [ ]:
sh('mkdir -p models/kokoro data/raw/esc50')
sh('wget -q -O models/kokoro/kokoro-v1.0.onnx https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx')
sh('wget -q -O models/kokoro/voices-v1.0.bin https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin')
sh('wget -q -O /tmp/esc50.zip https://github.com/karolpiczak/ESC-50/archive/master.zip && unzip -q -o /tmp/esc50.zip -d data/raw/esc50')

In [ ]:
# Build the split deterministically from seeds (CPU TTS; v1 takes ~10-15 min)
sh(f'cd src && python build_audio.py {SPLITS[SPLIT]} 2>&1 | tail -1')
sh(f'wc -l data/clinhallu/{SPLIT}/manifest.jsonl')  # pilot: 104, v1: 750

In [ ]:
# Evaluate, most important run first, so a cut-off still leaves useful results.
if SPLIT == 'v1' and PART == 2:  # model-free check: does VAD cut the start of utterances? (CPU, minutes)
    sh(f'cd src && python vad_audit.py {SPLIT}')
for m, p in RUNS:
    sh(f'cd src && python run_asr.py --split {SPLIT} --model {m} --preset {p} --device cuda --compute float16')
    sh(f'wc -l results/{SPLIT}/{m}__{p}.jsonl')

In [ ]:
# Zip only this split's new GPU results + its manifest (needed for local scoring)
OUT = f'{WORK}/results_{SPLIT}_part{PART}.zip'
files = ' '.join(f'results/{SPLIT}/{m}__{p}.jsonl' for m, p in RUNS)
if SPLIT == 'v1' and PART == 2: files += f' results/{SPLIT}/vad_audit.jsonl'
sh(f'zip -q {OUT} {files} data/clinhallu/{SPLIT}/manifest.jsonl')
sh(f'cd src && python aggregate.py {SPLIT} | head -25')
print('Download:', OUT, '(Kaggle: Output tab; Colab: Files panel)')